# Visualisation du Blueprint

Ce notebook montre comment visualiser le graphe de dépendances généré par le Blueprint.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from seapopym.blueprint import Blueprint
from seapopym.blueprint.nodes import DataNode, ComputeNode

bp = Blueprint()
bp.register_forcing("temperature")

# 1. Groupe Tuna
# compute_growth(temperature, nutrient) -> On va simplifier pour le test
# Disons compute_growth(temperature)
def grow_tuna(temperature): return temperature

bp.register_group("Tuna", [
    {
        "func": grow_tuna,
        "output_name": "biomass" # -> Tuna_biomass
    }
])

# 2. Groupe Shark
# Il a besoin de manger du thon.
# Sa fonction attend 'food'. On doit mapper 'food' -> 'Tuna_biomass'.
def grow_shark(food, temperature): return {"growth": food*(1 / temperature), "biomass": food*temperature}

bp.register_group("Shark", [
    {
        "func": grow_shark,
        "input_mapping": {"food": "Tuna_biomass"}, # Mapping explicite vers l'autre groupe
        # "output_name": "biomass" # -> Shark_biomass
        "output_name": {"temp_growth":"growth", "biomass":"biomass"} # -> Shark_biomass
    }
])

plan = bp.build()
print("Plan construit avec succès !")
print("Séquence :", [node.name for node in plan.task_sequence])

In [ ]:
# --- Visualisation ---
plt.figure(figsize=(12, 8))

G = bp.graph
pos = nx.spring_layout(G, seed=42)

# Séparation des noeuds par type
data_nodes = [n for n in G.nodes if isinstance(n, DataNode)]
compute_nodes = [n for n in G.nodes if isinstance(n, ComputeNode)]

# Dessin
nx.draw_networkx_nodes(G, pos, nodelist=data_nodes, node_color='lightblue', node_shape='o', label='Data')
nx.draw_networkx_nodes(G, pos, nodelist=compute_nodes, node_color='orange', node_shape='s', label='Compute')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20)

# Labels
labels = {n: n.name for n in G.nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=10)

plt.title("Blueprint Dependency Graph")
plt.legend()
plt.axis('off')
plt.show()